# SP24-BDS-030

# MUHAMMAD AZLAN

# Lab 02: PDFs, DOCX, and API Data Analysis

In [5]:
# !pip install PyPDF2 pdfplumber python-docx pandas nltk requests

import re
import csv
import json
import math
import requests
import pandas as pd
from collections import Counter
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

Defaulting to user installation because normal site-packages is not writeable


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\muham\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

In [15]:
PDF_PATH = r"E:\6th Sem\NLP\LAB 2\sample.pdf"

In [16]:
# (a) Extract text page by page
import pdfplumber

def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            pages.append(page.extract_text() or "")
    return pages

pages = extract_pages(PDF_PATH)
print("Total pages extracted:", len(pages))

Total pages extracted: 6


In [17]:
# (b) Merge pages, clean, tokenize
full_text = " ".join(pages)
cleaned = re.sub(r'[^a-zA-Z\s]', '', full_text).lower()
tokens = cleaned.split()
print("Total token count:", len(tokens))

Total token count: 814


In [18]:
# (c) Word frequency, excluding stopwords (20+)
extra_stopwords = {"also", "would", "could", "one", "may", "use", "used", "using"}
all_stops = STOPWORDS | extra_stopwords  # >20 stopwords removed
freq = Counter(t for t in tokens if t not in all_stops and len(t) > 1)
print("Top 10 words:")
for word, count in freq.most_common(10):
    print(f"  {word}: {count}")

Top 10 words:
  text: 13
  python: 10
  language: 9
  analysis: 9
  nlp: 8
  word: 7
  processing: 6
  like: 6
  natural: 5
  often: 5


In [19]:
# (d) Keyword search across pages, sentence by sentence
keyword = input("Enter a keyword to search for: ").strip().lower()
found = False
for i, page_text in enumerate(pages, start=1):
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    for sent in sentences:
        if keyword in sent.lower():
            print(f"[Page {i}] {sent.strip()}")
            found = True
if not found:
    print(f'No sentences found containing "{keyword}".')

Enter a keyword to search for:  1993


[Page 4] Working with PDF Documents
PDF, or Portable Document Format, was created by Adobe in 1993 and remains one of the most
widely used file formats for sharing documents.


## Task 2: Structured Page Metadata Extraction and CSV Export

In [21]:
PDF_PATH_2 = r"E:\6th Sem\NLP\LAB 2\sample.pdf"

In [22]:
# (a) Build pages_data
pages2 = extract_pages(PDF_PATH_2)

pages_data = []
for i, text in enumerate(pages2, start=1):
    text = text or ""
    word_count = len(text.split())
    char_count = len(text.replace(" ", "").replace("\n", ""))
    pages_data.append({
        "page_number": i,
        "raw_text": text,
        "word_count": word_count,
        "char_count": char_count
    })

print("First 2 entries:")
for entry in pages_data[:2]:
    print(entry)

First 2 entries:
{'page_number': 1, 'raw_text': 'Natural Language Processing with Python\nIntroduction to Natural Language Processing\nNatural Language Processing, often abbreviated as NLP, is a field of artificial intelligence that\nfocuses on the interaction between computers and human language. It was first explored in the\n1950s, and since 2010 it has grown rapidly due to advances in deep learning. Python is the most\npopular programming language for NLP work today, largely because of libraries such as NLTK\nand spaCy.\nResearchers estimate that by 2025 more than 80 percent of enterprise software will include\nsome form of natural language processing. This growth has been driven by applications such as\nchatbots, machine translation, sentiment analysis, and text summarization. You can learn more\nabout ongoing research at https://www.aclweb.org, which hosts papers from major NLP\nconferences.\nA typical NLP pipeline includes 5 major stages: text cleaning, tokenization, part-of-spee

In [23]:
# (b) Highest / lowest non-zero / average word count
non_zero = [p for p in pages_data if p["word_count"] > 0]

highest = max(pages_data, key=lambda p: p["word_count"])
print("Highest word count page:", highest["page_number"], "->", highest["word_count"], "words")
print(highest["raw_text"])
print("-" * 40)

if non_zero:
    lowest_nonzero = min(non_zero, key=lambda p: p["word_count"])
    print("Lowest non-zero word count page:", lowest_nonzero["page_number"], "->", lowest_nonzero["word_count"], "words")
else:
    print("No pages with non-zero word count.")

avg_words = round(sum(p["word_count"] for p in pages_data) / len(pages_data), 2) if pages_data else 0
print("Average word count across all pages:", avg_words)

Highest word count page: 1 -> 174 words
Natural Language Processing with Python
Introduction to Natural Language Processing
Natural Language Processing, often abbreviated as NLP, is a field of artificial intelligence that
focuses on the interaction between computers and human language. It was first explored in the
1950s, and since 2010 it has grown rapidly due to advances in deep learning. Python is the most
popular programming language for NLP work today, largely because of libraries such as NLTK
and spaCy.
Researchers estimate that by 2025 more than 80 percent of enterprise software will include
some form of natural language processing. This growth has been driven by applications such as
chatbots, machine translation, sentiment analysis, and text summarization. You can learn more
about ongoing research at https://www.aclweb.org, which hosts papers from major NLP
conferences.
A typical NLP pipeline includes 5 major stages: text cleaning, tokenization, part-of-speech
tagging, parsing, 

In [25]:
# (c) Search phrase across pages with a custom exception
class PhraseNotFoundError(Exception):
    pass

def search_phrase(pages_data, phrase):
    phrase_lower = phrase.lower()
    matches = []
    for p in pages_data:
        count = p["raw_text"].lower().count(phrase_lower)
        if count > 0:
            matches.append((p["page_number"], count))
    if not matches:
        raise PhraseNotFoundError(f'Phrase "{phrase}" was not found in any page.')
    return matches

search_term = input("Enter a search phrase: ").strip()
try:
    results = search_phrase(pages_data, search_term)
    for page_number, count in results:
        print(f"Page {page_number}: {count} occurrence(s)")
except PhraseNotFoundError as e:
    print("Error:", e)

Enter a search phrase:  1993


Page 4: 1 occurrence(s)


In [26]:
# (d) Write and re-read CSV with the csv module
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for p in pages_data:
        writer.writerow([p["page_number"], p["word_count"], p["char_count"]])

print("Re-reading pdf_summary.csv:")
with open("pdf_summary.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

Re-reading pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '174', '1000']
['2', '153', '812']
['3', '138', '790']
['4', '143', '805']
['5', '139', '812']
['6', '83', '458']


## Task 3: PDF Text Preprocessing Pipeline for NLP

In [27]:
PDF_PATH_3 = "sample.pdf"  # <-- set to your PDF file

In [28]:
# (a) Extract and clean text
pages3 = extract_pages(PDF_PATH_3)
raw_text3 = " ".join(pages3)
before_count = len(raw_text3)

step1 = re.sub(r'https?://\S+|www\.\S+', '', raw_text3)          # (i) remove URLs
step2 = re.sub(r'\d+', '', step1)                                # (ii) remove digits
step3 = re.sub(r'[^a-zA-Z\s.]', '', step2)                        # (iii) keep letters, spaces, full stops
cleaned3 = re.sub(r'\s+', ' ', step3).strip()                     # (iv) collapse spaces

after_count = len(cleaned3)
print("Characters before cleaning:", before_count)
print("Characters after cleaning:", after_count)

Characters before cleaning: 5506
Characters after cleaning: 5290


In [29]:
# (b) Lowercase, split into sentences, strip, remove empties
lower_text = cleaned3.lower()
raw_sentences = re.split(r'[.!?]', lower_text)
sentence_count_before = len(raw_sentences)

sentences3 = [s.strip() for s in raw_sentences if s.strip()]
sentence_count_after = len(sentences3)

print("Sentence count before removing empties:", sentence_count_before)
print("Sentence count after removing empties:", sentence_count_after)

Sentence count before removing empties: 43
Sentence count after removing empties: 42


In [30]:
# (c) Tokenize, remove stopwords, filter sentences with < 5 tokens left
before_filter = len(sentences3)
filtered_sentences = []
for sent in sentences3:
    tokens = sent.split()
    kept = [t for t in tokens if t not in STOPWORDS]
    if len(kept) >= 5:
        filtered_sentences.append(sent)
after_filter = len(filtered_sentences)

print("Sentence count before filter:", before_filter)
print("Sentence count after filter:", after_filter)
print("3 sample sentences:")
for s in filtered_sentences[:3]:
    print(" -", s)

Sentence count before filter: 42
Sentence count after filter: 42
3 sample sentences:
 - natural language processing with python introduction to natural language processing natural language processing often abbreviated as nlp is a field of artificial intelligence that focuses on the interaction between computers and human language
 - it was first explored in the s and since it has grown rapidly due to advances in deep learning
 - python is the most popular programming language for nlp work today largely because of libraries such as nltk and spacy


In [31]:
# (d) Write to file, re-read, report stats
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for s in filtered_sentences:
        f.write(s + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

print("Total lines written:", len(lines))
if lines:
    longest = max(lines, key=len)
    shortest = min(lines, key=len)
    print("Longest sentence:", longest)
    print("Shortest sentence:", shortest)

Total lines written: 42
Longest sentence: natural language processing with python introduction to natural language processing natural language processing often abbreviated as nlp is a field of artificial intelligence that focuses on the interaction between computers and human language
Shortest sentence: stopword removal is another common step


## Task 4: DOCX Paragraph Classification and Statistical Analysis

In [34]:
from docx import Document

DOCX_PATH_4 = "document_a.docx"  # <-- set to your Word document

In [35]:
# (a) Paragraph text + style, unique styles
doc4 = Document(DOCX_PATH_4)
para_records = [{"text": p.text, "style": p.style.name} for p in doc4.paragraphs]

print("Total paragraph count:", len(para_records))
unique_styles = sorted(set(r["style"] for r in para_records))
print("Unique style names:", unique_styles)

Total paragraph count: 9
Unique style names: ['Heading 1', 'Normal']


In [36]:
# (b) Classify into Headings / Body / Other-Empty
headings, body, other_empty = [], [], []
for r in para_records:
    style = r["style"]
    if style.startswith("Heading"):
        headings.append(r)
    elif style == "Normal" or "Body" in style:
        body.append(r)
    else:
        other_empty.append(r)

print("Headings:", len(headings))
print("Body:", len(body))
print("Other/Empty:", len(other_empty))

Headings: 4
Body: 5
Other/Empty: 0


In [37]:
# (c) Body paragraphs with more than 5 words
qualifying = []
for r in body:
    words = r["text"].split()
    if len(words) > 5:
        word_count = len(words)
        char_count = len(r["text"].replace(" ", ""))
        qualifying.append({"text": r["text"], "word_count": word_count, "char_count": char_count})

if qualifying:
    highest_wc = max(qualifying, key=lambda x: x["word_count"])
    lowest_wc = min(qualifying, key=lambda x: x["word_count"])
    mean_wc = round(sum(x["word_count"] for x in qualifying) / len(qualifying), 2)
    print("Highest word count paragraph:", highest_wc)
    print("Lowest word count paragraph (>5 words):", lowest_wc)
    print("Mean word count:", mean_wc)
else:
    print("No qualifying body paragraphs found.")

Highest word count paragraph: {'text': 'We collected survey data from over 2000 customers and combined it with support ticket logs to build a comprehensive view of the customer experience across all major regions.', 'word_count': 28, 'char_count': 146}
Lowest word count paragraph (>5 words): {'text': 'However, customers in the enterprise segment reported lower satisfaction with billing processes, citing confusing invoices and delayed refunds as their main concerns.', 'word_count': 22, 'char_count': 145}
Mean word count: 24.4


In [38]:
# (d) Nested dictionary by style, sorted by count desc, max 2 sample texts
style_dict = {}
for r in para_records:
    style = r["style"]
    style_dict.setdefault(style, {"count": 0, "texts": []})
    style_dict[style]["count"] += 1
    if len(style_dict[style]["texts"]) < 2:
        style_dict[style]["texts"].append(r["text"])

sorted_styles = dict(sorted(style_dict.items(), key=lambda kv: kv[1]["count"], reverse=True))

import pprint
pprint.pprint(sorted_styles)

{'Heading 1': {'count': 4, 'texts': ['Introduction', 'Methodology']},
 'Normal': {'count': 5,
            'texts': ['This report examines customer satisfaction trends '
                      'across our product lines over the last two fiscal '
                      'years, drawing on survey responses and support ticket '
                      'data.',
                      'We collected survey data from over 2000 customers and '
                      'combined it with support ticket logs to build a '
                      'comprehensive view of the customer experience across '
                      'all major regions.']}}


## Task 5: DOCX Table Extraction and Pandas Integration

In [39]:
DOCX_PATH_5 = "sample_with_table.docx"  # <-- set to a .docx with at least one table

In [40]:
# (a) Extract all tables as list of lists
doc5 = Document(DOCX_PATH_5)

all_tables = []
for table in doc5.tables:
    table_data = [[cell.text for cell in row.cells] for row in table.rows]
    all_tables.append(table_data)

print("Total tables found:", len(all_tables))
for i, t in enumerate(all_tables, start=1):
    rows = len(t)
    cols = len(t[0]) if rows else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

Total tables found: 1
Table 1: 6 rows x 4 columns


In [41]:
# (b) First table -> DataFrame, count empty/None cells per column
first_table = all_tables[0]
header, data_rows = first_table[0], first_table[1:]
df5 = pd.DataFrame(data_rows, columns=header)
print(df5)

empty_counts = {}
for col in df5.columns:
    empty_counts[col] = df5[col].apply(lambda v: v is None or str(v).strip() == "").sum()
print("Empty/None cells per column:", empty_counts)

          Name   Department Experience (yrs) Rating
0  Ayesha Khan  Engineering                5    4.5
1  Bilal Ahmed    Marketing                3       
2   Sara Malik                             7    4.8
3  Usman Tariq  Engineering                     3.9
4    Hina Raza        Sales                2    4.1
Empty/None cells per column: {'Name': np.int64(0), 'Department': np.int64(1), 'Experience (yrs)': np.int64(1), 'Rating': np.int64(1)}


In [42]:
# (c) Paragraphs before the first table
from docx.oxml.ns import qn

pre_table_paragraphs = []
for child in doc5.element.body:
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        text = "".join(node.text or "" for node in child.iter(qn('w:t')))
        pre_table_paragraphs.append(text)

print("Paragraphs before first table:", len(pre_table_paragraphs))
for p in pre_table_paragraphs:
    print(" -", p)

Paragraphs before first table: 12
 - Introduction to Data Science
 - Data science is a multidisciplinary field that uses scientific methods, algorithms, and systems to extract knowledge and insights from structured and unstructured data.
 - It combines statistics, computer science, and domain expertise to solve real world problems across industries such as healthcare, finance, and retail.
 - Key Skills for Data Scientists
 - A strong data scientist typically has skills in programming languages like Python and R, as well as a solid understanding of statistics and machine learning algorithms.
 - Short note.
 - 
 - Machine Learning Overview
 - Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data without being explicitly programmed for every scenario.
 - Common types of machine learning include supervised learning, unsupervised learning, and reinforcement learning, each suited to different kinds of problems and available data.
 - Sample D

In [43]:
# (d) Write pre-table paragraphs + first table data to docx_extracted.txt
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for p in pre_table_paragraphs:
        f.write(p + "\n")
    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("Wrote docx_extracted.txt")
with open("docx_extracted.txt", "r", encoding="utf-8") as f:
    print(f.read())

Wrote docx_extracted.txt
--- PRE-TABLE PARAGRAPHS ---
Introduction to Data Science
Data science is a multidisciplinary field that uses scientific methods, algorithms, and systems to extract knowledge and insights from structured and unstructured data.
It combines statistics, computer science, and domain expertise to solve real world problems across industries such as healthcare, finance, and retail.
Key Skills for Data Scientists
A strong data scientist typically has skills in programming languages like Python and R, as well as a solid understanding of statistics and machine learning algorithms.
Short note.

Machine Learning Overview
Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data without being explicitly programmed for every scenario.
Common types of machine learning include supervised learning, unsupervised learning, and reinforcement learning, each suited to different kinds of problems and available data.
Sample Dataset Overvi

## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

In [44]:
DOCX_PATH_A = "document_a.docx"  # <-- Document A
DOCX_PATH_B = "document_b.docx"  # <-- Document B

In [45]:
# (a) Headings from each document, set operations
doc_a = Document(DOCX_PATH_A)
doc_b = Document(DOCX_PATH_B)

headings_a = {p.text.strip() for p in doc_a.paragraphs if p.style.name.startswith("Heading")}
headings_b = {p.text.strip() for p in doc_b.paragraphs if p.style.name.startswith("Heading")}

common = headings_a & headings_b
only_a = headings_a - headings_b
only_b = headings_b - headings_a

print("Common headings:", common)
print("Only in Document A:", only_a)
print("Only in Document B:", only_b)

Common headings: {'Results', 'Introduction'}
Only in Document A: {'Conclusion', 'Methodology'}
Only in Document B: {'Survey Design', 'Recommendations'}


In [46]:
# (b) Normal-style paragraphs, merge, deduplicate
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip()]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip()]

combined = normal_a + normal_b
count_before = len(combined)

seen = set()
deduped = []
for text in combined:
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        deduped.append(text)

count_after = len(deduped)
print("Count before deduplication:", count_before)
print("Count after deduplication:", count_after)

Count before deduplication: 10
Count after deduplication: 9


In [47]:
# (c) Word frequency across deduplicated paragraphs
extra_stops_6 = {"also", "would", "could", "one", "may", "like", "well",
                  "get", "got", "us", "said", "even"}  # >= 15 stopwords removed overall
all_stops_6 = STOPWORDS | extra_stops_6

tokens_6 = []
for text in deduped:
    cleaned = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    tokens_6.extend(cleaned.split())

freq_6 = Counter(t for t in tokens_6 if t not in all_stops_6 and len(t) > 1)
print("Top 20 words:")
for word, count in freq_6.most_common(20):
    print(f"  {word}: {count}")

Top 20 words:
  satisfaction: 5
  support: 5
  customer: 4
  survey: 4
  across: 3
  customers: 3
  reported: 3
  engagement: 3
  ticket: 2
  data: 2
  collected: 2
  overall: 2
  scores: 2
  year: 2
  largely: 2
  enterprise: 2
  billing: 2
  improvements: 2
  employees: 2
  report: 1


In [48]:
# (d) Summary statistics
total_headings = len(headings_a) + len(headings_b)
total_unique_body = len(deduped)
vocab_size = len(freq_6)
avg_para_len = round(sum(len(t.split()) for t in deduped) / len(deduped), 2) if deduped else 0
longest_para = max(deduped, key=lambda t: len(t.split())) if deduped else None

print("Total headings (both docs):", total_headings)
print("Total unique body paragraphs:", total_unique_body)
print("Vocabulary size:", vocab_size)
print("Average paragraph length (words):", avg_para_len)
print("Longest paragraph:", longest_para)

Total headings (both docs): 8
Total unique body paragraphs: 9
Vocabulary size: 112
Average paragraph length (words): 23.78
Longest paragraph: We collected survey data from over 2000 customers and combined it with support ticket logs to build a comprehensive view of the customer experience across all major regions.


## Task 7: Multi-Endpoint API Data Joining and Analysis

In [49]:
# (a) Fetch users and posts, verify status 200
users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_resp = requests.get(users_url)
if users_resp.status_code != 200:
    raise ConnectionError(f"Failed to fetch users: status {users_resp.status_code}")
users = users_resp.json()

posts_resp = requests.get(posts_url)
if posts_resp.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts: status {posts_resp.status_code}")
posts = posts_resp.json()

print("Users fetched:", len(users))
print("Posts fetched:", len(posts))

Users fetched: 10
Posts fetched: 100


In [50]:
# (b) Join users and posts
joined = {}
for user in users:
    user_posts = [p for p in posts if p["userId"] == user["id"]]
    joined[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": [p["title"] for p in user_posts]
    }

for name in list(joined.keys())[:3]:
    print(name, "->", joined[name])

Leanne Graham -> {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}
Ervin Howell -> {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adipisci placeat illum aut reiciendis qui', 'doloribus ad provident suscipit at']}
Clementine Bauch -> {'email

In [51]:
# (c) User with the most posts
top_user_name = max(joined, key=lambda n: joined[n]["post_count"])
top_user_info = joined[top_user_name]
top_user_full = next(u for u in users if u["name"] == top_user_name)

print("Name:", top_user_name)
print("Email:", top_user_info["email"])
print("Company:", top_user_full["company"]["name"])
print("Post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"  {i}. {title}")

Name: Leanne Graham
Email: Sincere@april.biz
Company: Romaguera-Crona
Post titles:
  1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
  2. qui est esse
  3. ea molestias quasi exercitationem repellat qui ipsa sit aut
  4. eum et est occaecati
  5. nesciunt quas odio
  6. dolorem eum magni eos aperiam quia
  7. magnam facilis autem
  8. dolorem dolore est ipsam
  9. nesciunt iure omnis dolorem tempora et accusantium
  10. optio molestias id quia eum


In [52]:
# (d) Filter users by company.bs containing "synergies" or city longer than 8 chars
matches_7 = []
for user in users:
    bs = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in bs or len(city) > 8:
        post_count = joined[user["name"]]["post_count"]
        matches_7.append((user["name"], city, post_count))

print("Matching users:")
for name, city, post_count in matches_7:
    print(f"  {name} | {city} | posts: {post_count}")

Matching users:
  Leanne Graham | Gwenborough | posts: 10
  Ervin Howell | Wisokyburgh | posts: 10
  Clementine Bauch | McKenziehaven | posts: 10
  Patricia Lebsack | South Elvis | posts: 10
  Chelsey Dietrich | Roscoeview | posts: 10
  Mrs. Dennis Schulist | South Christy | posts: 10
  Kurtis Weissnat | Howemouth | posts: 10
  Nicholas Runolfsdottir V | Aliyaview | posts: 10
  Glenna Reichert | Bartholomebury | posts: 10
  Clementina DuBuque | Lebsackbury | posts: 10


## Task 8: To-Do Completion Rate Analysis Across Users

In [53]:
# (a) Fetch and validate todos
todos_url = "https://jsonplaceholder.typicode.com/todos"
todos_resp = requests.get(todos_url)

if todos_resp.status_code != 200:
    print("Error: failed to fetch todos, status", todos_resp.status_code)
else:
    todos = todos_resp.json()
    if not isinstance(todos, list):
        print("Error: response is not a list")
    elif len(todos) == 0:
        print("Error: no todo records found")
    else:
        print("Total todo records retrieved:", len(todos))

Total todo records retrieved: 200


In [54]:
# (b) Group todos by userId, verify completed + incomplete == total
user_todo_stats = {}
for t in todos:
    uid = t["userId"]
    user_todo_stats.setdefault(uid, {"total": 0, "completed": 0, "incomplete": 0})
    user_todo_stats[uid]["total"] += 1
    if t["completed"]:
        user_todo_stats[uid]["completed"] += 1
    else:
        user_todo_stats[uid]["incomplete"] += 1

for uid in sorted(user_todo_stats):
    stats = user_todo_stats[uid]
    is_consistent = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} -> consistent: {is_consistent}")

User 1: {'total': 20, 'completed': 11, 'incomplete': 9} -> consistent: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12} -> consistent: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13} -> consistent: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14} -> consistent: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8} -> consistent: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14} -> consistent: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11} -> consistent: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9} -> consistent: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12} -> consistent: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8} -> consistent: True


In [55]:
# (c) Completion rate per user
completion_rates = {
    uid: round(stats["completed"] / stats["total"] * 100, 1)
    for uid, stats in user_todo_stats.items()
}

best_uid = max(completion_rates, key=completion_rates.get)
worst_uid = min(completion_rates, key=completion_rates.get)
overall_avg = round(sum(completion_rates.values()) / len(completion_rates), 1)

print("Highest completion rate: user", best_uid, "->", completion_rates[best_uid], "%")
print("Lowest completion rate: user", worst_uid, "->", completion_rates[worst_uid], "%")
print("Overall average completion rate:", overall_avg, "%")

Highest completion rate: user 5 -> 60.0 %
Lowest completion rate: user 4 -> 30.0 %
Overall average completion rate: 45.0 %


In [56]:
# (d) Ranked table with user names
users_lookup_resp = requests.get(users_url)
users_lookup = {u["id"]: u["name"] for u in users_lookup_resp.json()}

ranking = sorted(completion_rates.items(), key=lambda kv: kv[1], reverse=True)

header = f"{'Rank':<5}{'Name':<25}{'Total Tasks':<13}{'Completed':<11}{'Completion Rate %':<18}"
print(header)
print("-" * len(header))
for rank, (uid, rate) in enumerate(ranking, start=1):
    name = users_lookup.get(uid, f"User {uid}")
    total = user_todo_stats[uid]["total"]
    completed = user_todo_stats[uid]["completed"]
    print(f"{rank:<5}{name:<25}{total:<13}{completed:<11}{rate:<18}")

Rank Name                     Total Tasks  Completed  Completion Rate % 
------------------------------------------------------------------------
1    Chelsey Dietrich         20           12         60.0              
2    Clementina DuBuque       20           12         60.0              
3    Leanne Graham            20           11         55.0              
4    Nicholas Runolfsdottir V 20           11         55.0              
5    Kurtis Weissnat          20           9          45.0              
6    Ervin Howell             20           8          40.0              
7    Glenna Reichert          20           8          40.0              
8    Clementine Bauch         20           7          35.0              
9    Patricia Lebsack         20           6          30.0              
10   Mrs. Dennis Schulist     20           6          30.0              


## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [57]:
# (a) Flatten user objects
users_resp_9 = requests.get(users_url)
users_9 = users_resp_9.json()

def flatten_user(user: dict) -> dict:
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"],
    }

flattened_users = [flatten_user(u) for u in users_9]
print(flattened_users)

[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}, {'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 'lng': '-164.2990

In [58]:
# (b) DataFrame + distance_from_origin
df9 = pd.DataFrame(flattened_users)
df9["lat"] = df9["lat"].astype(float)
df9["lng"] = df9["lng"].astype(float)
df9["distance_from_origin"] = df9.apply(lambda row: math.sqrt(row["lat"]**2 + row["lng"]**2), axis=1)

pd.set_option("display.max_columns", None)
print(df9)

farthest = df9.loc[df9["distance_from_origin"].idxmax()]
print("\nUser with greatest distance from origin:")
print(farthest)

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city     zipcode      lat       lng        company_name  \
0     Gwenborough  92998-3874 -37.3159   81.149

In [59]:
# (c) Hemisphere classification
northern = df9[df9["lat"] > 0]["name"].tolist()
southern = df9[df9["lat"] < 0]["name"].tolist()

print("Northern Hemisphere users:", northern)
print("Southern Hemisphere users:", southern)

avg_dist_north = df9[df9["lat"] > 0]["distance_from_origin"].mean()
avg_dist_south = df9[df9["lat"] < 0]["distance_from_origin"].mean()

print("Average distance (Northern):", round(avg_dist_north, 2))
print("Average distance (Southern):", round(avg_dist_south, 2))
print("Higher average distance hemisphere:", "Northern" if avg_dist_north > avg_dist_south else "Southern")

Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']
Average distance (Northern): 123.58
Average distance (Southern): 84.32
Higher average distance hemisphere: Northern


In [60]:
# (d) Comment counts per post
posts_resp_9 = requests.get("https://jsonplaceholder.typicode.com/posts")
comments_resp_9 = requests.get("https://jsonplaceholder.typicode.com/comments")
posts_9 = posts_resp_9.json()
comments_9 = comments_resp_9.json()

comment_counts = Counter(c["postId"] for c in comments_9)

post_comment_list = [
    (p["id"], p["title"][:40], comment_counts.get(p["id"], 0))
    for p in posts_9
]
post_comment_list.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most-commented posts:")
for row in post_comment_list[:5]:
    print(row)

print("\nBottom 5 least-commented posts:")
for row in post_comment_list[-5:]:
    print(row)

Top 5 most-commented posts:
(1, 'sunt aut facere repellat provident occae', 5)
(2, 'qui est esse', 5)
(3, 'ea molestias quasi exercitationem repell', 5)
(4, 'eum et est occaecati', 5)
(5, 'nesciunt quas odio', 5)

Bottom 5 least-commented posts:
(96, 'quaerat velit veniam amet cupiditate aut', 5)
(97, 'quas fugiat ut perspiciatis vero provide', 5)
(98, 'laboriosam dolor voluptates', 5)
(99, 'temporibus sit alias delectus eligendi p', 5)
(100, 'at nam consequatur ea labore ea harum', 5)
